# Session 3 — Data analysis with pandas
*Séance 3 — Analyse de données avec pandas*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/ML4EO_Bootcamp/blob/main/notebooks/03_pandas_meteo.ipynb)

**Course:** Machine Learning for Earth Observation (45 h) · **Instructor:** Noé Carème Fouotsa Manfouo

| Training track · Parcours de formation | This session · Cette séance |
|---|---|
| Geomatics / environment · Géomatique / environnement (UdeS géomatique appliquée; Bishop's Environmental Studies & Geography) | ● full session / séance complète |
| Computer science · Informatique (UdeS informatique; Bishop's Computer Science) | ● full session / séance complète |
| College / continuing education · Cégep / formation continue | ● full session / séance complète |

**By the end of this session you will be able to:**

1. Read a messy CSV file with pandas, handling no-data codes.
2. Select, filter and summarize columns; convert units with vectorized operations.
3. Parse dates and aggregate a time series by month.

**How to use this notebook:** read the lecture cells, run every code cell (`Shift + Enter`), then complete the cells marked
`# YOUR CODE HERE` in the lab section. Run the ✅ check cells to get instant feedback.

In [ ]:
# --- Setup (run once) -------------------------------------------------------
import sys, pathlib, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not pathlib.Path("ML4EO_Bootcamp").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/ML4EO_Bootcamp.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *"geopandas rasterio rioxarray".split()], check=True)
    ROOT = pathlib.Path("ML4EO_Bootcamp")
else:
    ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA = ROOT / "data"
sys.path.append(str(ROOT / "notebooks"))   # course helper module (ml4eo_data.py)


def check(condition, message="Check failed"):
    """Instant feedback: raises an error with a hint if the condition is False."""
    assert condition, message
    print("✅ Passed")


def check_equal(value, expected, message=None):
    assert value == expected, message or f"Expected {expected!r}, got {value!r}"
    print("✅ Passed")

## Part A — Lecture: pandas essentials

We use hourly weather observations from the **NOAA Integrated Surface Database** for a station in Finland (May–Aug 2017).
Missing values are coded with asterisks (`*`, `**`, `***`…), described in `data/weather/3505doc.txt`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

fp = DATA / "weather" / "6153237444115dat.csv"
data = pd.read_csv(fp, na_values=['*', '**', '***', '****', '*****', '******'], low_memory=False)
data.head()

In [ ]:
print(data.shape)          # rows, columns
data[['TEMP', 'MAX', 'MIN']].describe()

**Selecting and filtering** — a column is a `Series`; boolean conditions select rows.

In [ ]:
temp_f = data['TEMP']
hot = data[data['TEMP'] > 70]
print(len(hot), "hourly records above 70 °F")

**Vectorized operations** apply to the whole column at once (no loop).

In [ ]:
data['TEMP_C'] = (data['TEMP'] - 32) / 1.8
data[['TEMP', 'TEMP_C']].head()

**Dates and aggregation** — parse the `YR--MODAHRMN` column, then resample.

In [ ]:
data['time'] = pd.to_datetime(data['YR--MODAHRMN'].astype(str), format='%Y%m%d%H%M')
daily = data.set_index('time')['TEMP_C'].resample('D').mean()
daily.plot(figsize=(9, 3), title='Daily mean temperature (°C)')
plt.ylabel('°C'); plt.show()

## Part B — Lab

**Task 1 — Explore.** Store the number of rows in `rows`, the column names (a list) in `column_names`, and the number of
unique stations (column `USAF`) in `station_count`.

In [ ]:
# YOUR CODE HERE
rows = None
column_names = None
station_count = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check_equal(rows, 11694)
check('TEMP' in column_names)
check_equal(station_count, 2)

**Task 2 — Statistics.** Compute `temp_mean` (mean of `TEMP`, °F) and `temp_max_std` (standard deviation of `MAX`).

In [ ]:
# YOUR CODE HERE
temp_mean = None
temp_max_std = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(round(temp_mean, 1) == round(data['TEMP'].mean(), 1))
check(temp_max_std > 0)

**Task 3 — Select and convert.** Create `selected` with columns `USAF`, `time`, `TEMP`, `MAX`, `MIN`, drop rows where `TEMP` is
missing, and add a column `Celsius` (rounded to 0 decimals, integer type) from `TEMP`.

In [ ]:
# YOUR CODE HERE
selected = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(selected['TEMP'].isna().sum() == 0, "Drop missing TEMP values")
check(str(selected['Celsius'].dtype).startswith('int'))

**Task 4 — Monthly summary.** For each station (`USAF`), compute the **monthly mean** Celsius temperature into a DataFrame
`monthly` with one row per station and month (hint: `groupby` with `selected['time'].dt.month`).

In [ ]:
# YOUR CODE HERE
monthly = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check_equal(len(monthly), 8, "2 stations × 4 months")

**❓ Question —** Which month was warmest at each station? Is the difference between the two stations plausible given their location?

*Your answer here.*

## Summary

- `read_csv(na_values=…)` turns no-data codes into NaN at read time.
- Boolean masks, vectorized arithmetic and `groupby` replace most loops.
- Time series are parsed with `to_datetime` and summarized with `resample` or `groupby`.

**To go further:**

- pandas user guide: https://pandas.pydata.org/docs/user_guide/

---
*Lecture and exercises adapted from the Geo-Python course, © The Geo-Python team, Department of Geosciences and Geography, University of Helsinki, CC BY-SA 4.0 (code GPLv3) — https://geo-python-site.readthedocs.io. Modified by N. C. Fouotsa Manfouo; this adaptation is shared under the same licence.*